# Baseline-решение

По мотивам ноутбука https://www.kaggle.com/code/hardtype/parsing-news-from-rbc-lenta-ru

## 1. Парсим новости с сайта Lenta.ru

In [ ]:
# Установка библиотек
!pip install bs4
!pip install openpyxl
!pip install webdriver-manager

!apt-get update -q
!apt-get install -y chromium-browser chromium-chromedriver -q
!pip install google_colab_selenium -q

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 https://cli.github.com/packages stable InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,201 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/r

In [ ]:
# Импорт библиотек
import requests as rq
from bs4 import BeautifulSoup as bs
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
# from IPython import display

# Запуск браузера
# Настройки WebDriver
# Если запускаете локально

# chrome_options = webdriver.ChromeOptions()
# chrome_options.add_argument("--blink-settings=imagesEnabled=false")
# chrome_options.add_argument("headless")
# chrome_options.add_argument("no-sandbox")
# chrome_options.add_argument("disable-dev-shm-usage")
# driver = webdriver.Chrome(options=chrome_options)


# Запуск браузера
# Настройки WebDriver
# Если запускаете в коллаб

import google_colab_selenium as gs
driver = gs.Chrome()

<IPython.core.display.Javascript object>

In [ ]:
  # url = 'https://lenta.ru/search/v2/process?'
  #       + 'from=0&'\                       # Смещение
  #       + 'size=1000&'\                    # Кол-во статей
  #       + 'sort=2&'\                       # Сортировка по дате (2), по релевантности (1)
  #       + 'title_only=0&'\                 # Точная фраза в заголовке
  #       + 'domain=1&'\                     # ??
  #       + 'modified%2Cformat=yyyy-MM-dd&'\ # Формат даты
  #       + 'type=1&'\                       # Материалы. Все материалы (0). Новость (1)
  #       + 'bloc=4&'\                       # Рубрика. Экономика (4). Все рубрики (0)
  #       + 'modified%2Cfrom=2020-01-01&'\
  #       + 'modified%2Cto=2020-11-01&'\
  #       + 'query='                         # Поисковой запрос

In [ ]:
import pandas as pd
import requests as rq
from datetime import datetime, timedelta
from IPython import display


class lentaRu_parser:
    def __init__(self):
        pass

    def _get_url(self, param_dict: dict) -> str:
        """
        Возвращает URL для запроса json таблицы со статьями
        """
        hasType = int(param_dict['type']) != 0
        hasBloc = int(param_dict['bloc']) != 0

        url = (
            'https://lenta.ru/search/v2/process?'
            + 'from={}&'.format(param_dict['from'])
            + 'size={}&'.format(param_dict['size'])
            + 'sort={}&'.format(param_dict['sort'])
            + 'title_only={}&'.format(param_dict['title_only'])
            + 'domain={}&'.format(param_dict['domain'])
            + 'modified%2Cformat=yyyy-MM-dd&'
        )

        # Добавляем условные параметры только если они нужны
        if hasType:
            url += 'type={}&'.format(param_dict['type'])
        if hasBloc:
            url += 'bloc={}&'.format(param_dict['bloc'])

        url += (
            'modified%2Cfrom={}&'.format(param_dict['dateFrom'])
            + 'modified%2Cto={}&'.format(param_dict['dateTo'])
            + 'query={}'.format(param_dict['query'])
        )

        return url

    def _get_search_table(self, param_dict: dict) -> pd.DataFrame:
        """
        Возвращает pd.DataFrame со списком статей
        """
        url = self._get_url(param_dict)
        r = rq.get(url)
        r.raise_for_status()  # полезно для явной обработки ошибок
        search_table = pd.DataFrame(r.json()['matches'])
        return search_table

    def get_articles(
        self,
        param_dict,
        time_step=37,
        save_every=5,
        save_excel=True
    ) -> pd.DataFrame:
        """
        Функция для скачивания статей интервалами через каждые time_step дней
        Делает сохранение таблицы через каждые save_every * time_step дней
        """
        param_copy = param_dict.copy()
        time_step = timedelta(days=time_step)
        dateFrom = datetime.strptime(param_copy['dateFrom'], '%Y-%m-%d')
        dateTo = datetime.strptime(param_copy['dateTo'], '%Y-%m-%d')
        if dateFrom > dateTo:
            raise ValueError('dateFrom should be less than dateTo')

        out = pd.DataFrame()
        save_counter = 0

        while dateFrom <= dateTo:
            param_copy['dateTo'] = (dateFrom + time_step).strftime('%Y-%m-%d')
            if dateFrom + time_step > dateTo:
                param_copy['dateTo'] = dateTo.strftime('%Y-%m-%d')

            print(
                'Parsing articles from '
                + param_copy['dateFrom'] + ' to ' + param_copy['dateTo']
            )

            chunk_df = self._get_search_table(param_copy)

            out = pd.concat([out, chunk_df], ignore_index=True)

            dateFrom += time_step + timedelta(days=1)
            param_copy['dateFrom'] = dateFrom.strftime('%Y-%m-%d')
            save_counter += 1

            if save_counter == save_every:
                display.clear_output(wait=True)
                out.to_excel("/tmp/checkpoint_table.xlsx", index=False)
                print('Checkpoint saved!')
                save_counter = 0

        if save_excel:
            out.to_excel(
                "lenta_{}_{}.xlsx".format(
                    param_dict['dateFrom'], param_dict['dateTo']
                ),
                index=False
            )
        print('Finish')
        return out


In [ ]:
# Задаем тут параметры
query = 'экономика'
offset = 0
size = 1000
sort = "3"
title_only = "0"
domain = "1"
material = "0"
bloc = "0" # topic = тематика новости
dateFrom = '2023-01-01'
dateTo = "2023-12-30"

param_dict = {'query'     : query,
              'from'      : str(offset),
              'size'      : str(size),
              'dateFrom'  : dateFrom,
              'dateTo'    : dateTo,
              'sort'      : sort,
              'title_only': title_only,
              'type'      : material,
              'bloc'      : bloc,
              'domain'    : domain}

print("param_dict:", param_dict)

param_dict: {'query': 'экономика', 'from': '0', 'size': '1000', 'dateFrom': '2023-01-01', 'dateTo': '2023-12-30', 'sort': '3', 'title_only': '0', 'type': '0', 'bloc': '0', 'domain': '1'}


In [ ]:
# Тоже будем собирать итеративно, правда можно ставить time_step побольше, т.к.
# больше лимит на запрос статей. И Работает быстрее :)

parser = lentaRu_parser()

tbl = parser.get_articles(param_dict=param_dict,
                         time_step = 37,
                         save_every = 5,
                         save_excel = True)
print(len(tbl.index))
tbl.head()

Checkpoint saved!
Finish
5123


,docid,url,title,modified,lastmodtime,type,domain,status,part,bloc,tags,image_url,pubdate,text,rightcol,snippet
0,1363816,https://lenta.ru/news/2023/01/01/cred/,ЦБ установил ограничения на выдачу кредитов ро...,1672538964,1672538964,1,1,0,0,4,[198],https://icdn.lenta.ru/images/2023/01/01/05/202...,1672538964,Фото: Михаил Воскресенский / РИА Новости Марин...,ЦБ установил ограничения на выдачу кредитов ро...,Дальнейший же рост закредитованности ... «в ус...
1,1363819,https://lenta.ru/news/2023/01/01/sk/,Президент Южной Кореи пообещал сосредоточить д...,1672539535,1672539535,1,1,0,0,2,[1],https://icdn.lenta.ru/images/2023/01/01/05/202...,1672539535,Юн Сок Ель Фото: Kim Jae-Hwan / Globallookpres...,Президент Южной Кореи пообещал сосредоточить д...,Юн Сок Ель Фото: Kim Jae-Hwan / ... сосредоточ...
2,1363827,https://lenta.ru/news/2023/01/01/jjapan_/,В Японии заявили о нештатной ситуации в товаро...,1672543020,1672548028,1,1,0,0,2,[1],https://icdn.lenta.ru/images/2023/01/01/07/202...,1672543020,Фото: Victor Lisitsyn / Global Look Press Мари...,В Японии заявили о нештатной ситуации в товаро...,Фото: Victor Lisitsyn / Global Look ... россий...
3,1352389,https://lenta.ru/news/2023/01/01/saveyourmoney/,Россиянам назвали прибыльные варианты вложений...,1672547901,1672547901,1,1,0,0,4,[272],https://icdn.lenta.ru/images/2022/12/09/20/202...,1672547901,Фото: Mike Segar / Reuters Мария Сметанина Руб...,Россиянам назвали прибыльные варианты вложений...,"Они, в первую очередь золото, будут расти, пот..."
4,1363839,https://lenta.ru/news/2023/01/01/war_econom/,Путин заявил об объявленной санкционной войне ...,1672553520,1672553946,1,1,0,0,1,[1],https://icdn.lenta.ru/images/2023/01/01/09/202...,1672553520,Фото: Алексей Даничев / РИА Новости Варвара Ко...,Путин заявил об объявленной санкционной войне ...,"Путин добавил, что власти страны делают ... су..."


In [ ]:
tbl.to_csv("Lenta_sample.csv", index=False)

In [ ]:
# tbl = pd.read_csv("Lenta_sample.csv")

In [ ]:
tbl.shape

In [ ]:
tbl['bloc'].value_counts(normalize=True)

,proportion
bloc,
4,0.492485
2,0.186805
1,0.124341
3,0.073785
7,0.032793
12,0.032208
0,0.018739
48,0.011712
5,0.008198


Найдем соответствие между кодом блока, его названием и кодом в соревновании:

* 1 - Россия - 0
* 37 - Силовые структуры - 2
* 3 - Бывший СССР - 3
* 4 - Экономика - 1
* 5 - Наука и техника - 8
* 8 - Спорт - 4
* 48 - Туризм - 7
* 87 - Здоровье - 5

In [ ]:
tbl[tbl.bloc == 3].iloc[0]

In [ ]:
tbl = tbl[tbl.bloc.isin([1, 37, 3, 4, 5, 8, 48, 87])]

TagsMap = {1 : 0, 3 : 3, 4 : 1, 5 : 8, 8 : 4, 37 : 2, 48 : 7, 87 : 5}

tbl['topic'] = tbl['bloc'].map(TagsMap)

In [ ]:
tbl.shape

In [ ]:
tbl['topic'].value_counts(normalize=True) # можно сверить с распределением меток классов в соревновании

## 2. Машинное обучение

Загружаем данные и обучаем модель на разбиении трейн-тест

In [ ]:
tbl_new = tbl[~tbl.text.isna()]

print(len(tbl), len(tbl_new))

In [ ]:
X = tbl_new[['text']]
y = tbl_new['topic']

X.shape

In [ ]:
# использовать "вероятностные модели"

# class_weight = "balanced"

# в обучающих данных взять поровну новостей каждого класса

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
X_train.shape, X_test.shape

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vec = CountVectorizer() # подбор гиперпараметров очень помогает
vec.fit(X_train['text'])

bow = vec.transform(X_train['text'])  # bow — bag of words (мешок слов)
bow_test = vec.transform(X_test['text'])

print(bow.shape)

scaler = MaxAbsScaler()
bow = scaler.fit_transform(bow)
bow_test = scaler.transform(bow_test)

clf = LogisticRegression(max_iter=200, random_state=42)
clf.fit(bow, y_train)
pred = clf.predict(bow_test)

print(classification_report(y_test, pred))

Загружаем тестовые данные, обучаем итоговую модель и делаем прогноз.

In [ ]:
Test = pd.read_csv("test_news.csv")
Test

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vec = CountVectorizer()
vec.fit(X['text'])

bow = vec.transform(X['text'])  # bow — bag of words (мешок слов)
bow_test = vec.transform(Test['content'])

scaler = MaxAbsScaler()
bow = scaler.fit_transform(bow)
bow_test = scaler.transform(bow_test)

clf = LogisticRegression(max_iter=200, random_state=42)
clf.fit(bow, y)
pred = clf.predict(bow_test)

In [ ]:
pred[:10], len(pred)

Сохраняем прогноз в файл.

In [ ]:
subm = pd.read_csv("base_submission_news.csv")
subm.head()

In [ ]:
subm['topic'] = pred

subm.to_csv("bow_logreg_lenta.csv", index=False)

In [ ]:
subm['topic'].value_counts(normalize=True)